In [ ]:
import os
from Bio import SeqIO
from Bio.SeqRecord import SeqRecord
from Bio.Seq import Seq
from go_ml.eval_utils import filter_annot_df
import pandas as pd

# Run from go_ml/msa_pipeline/
data_root = '../gen_datasets/datasets'
fasta_out_dir = '../gen_datasets/datasets'

# Datasets that have MSA baselines
msa_dataset_labels = ['csa', 'llps', 'elms', 'ip_domain']
dataset_dfs = {
    label: filter_annot_df(pd.read_csv(f'{data_root}/{label}_dataset.csv', sep='\t'))
    for label in msa_dataset_labels
}

In [ ]:
# Write one FASTA per dataset.
# csa/llps/elms use UniprotID as sequence IDs (matching the m8 query IDs).
# ip_domain uses ipdom{i} IDs (integer index) because some UniprotIDs contain
# colons that cause issues downstream.

for label, df in dataset_dfs.items():
    out_path = f'{fasta_out_dir}/{label}.fasta'
    if label == 'ip_domain':
        records = [SeqRecord(Seq(seq), id=f'ipdom{i}', description='')
                   for i, seq in enumerate(df['Sequence'])]
    else:
        records = [SeqRecord(Seq(seq), id=uid, description='')
                   for uid, seq in zip(df['UniprotID'], df['Sequence'])]
    SeqIO.write(records, out_path, 'fasta')
    print(f'Wrote {len(records)} sequences to {out_path}')

179